# Dalitz + discriminating variables

This example keeps the specialized factorized likelihood, but the Dalitz signal/background samples are generated with the high-level toy helpers. The mass and BDT toys are sampled from the same PDFs used in the fit, giving a true closure example. One-dimensional data projections follow the project convention: black circular points with statistical error bars.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BackgroundCategory, DecayChannel, DecayModel, Exponential1D, FactorizedDensity,
    Gaussian1D, Histogram1D, Minimizer, MultiBackgroundNLL, NonResonant,
    Parameter, RealImag, Resonance, ToyBackground, enable_x64, generate_toy,
    plot_binned_data, plot_dalitz,
)
from dalitzplotfitter.background import FunctionalBackground

enable_x64()


In [ ]:
model = DecayModel(
    DecayChannel("B+", ("K+", "pi+", "pi-")),
    [
        Resonance("Kstar892", (0,2), RealImag(1,0),
                  mass=0.8958, width=0.0474, spin=1),
        Resonance("rho770", (1,2), RealImag(0.65,0.10),
                  mass=0.7753, width=0.1491, spin=1),
        NonResonant(RealImag(-0.5,0.1)),
    ],
    normalization_method="square-dalitz",
    normalization_resolution=180,
    normalization_pair=(0,2),
)
norm = model.normalization_sample
bkg_shape = FunctionalBackground(
    lambda d: 0.4 + 0.8*(d["s13"]-jnp.min(norm.s13))/(jnp.max(norm.s13)-jnp.min(norm.s13))
)

N = 20_000
FS_TRUE = 0.70
ns = int(round(N*FS_TRUE))
nb = N-ns
sig = generate_toy(model, ns, seed=10001, pool_size=120_000)
bkg = generate_toy(
    model, nb, signal_fraction=0.0,
    backgrounds=(ToyBackground("combinatorial", bkg_shape),),
    seed=10002, pool_size=120_000,
)

def merge(a, b):
    kwargs = {
        name: jnp.concatenate((getattr(a,name), getattr(b,name)))
        for name in ("s12","s13","s23")
    }
    return type(a)(**kwargs, weights=jnp.ones(a.size+b.size))

data = merge(sig,bkg)


In [ ]:
rng = np.random.default_rng(10003)

def truncated_normal(mean, sigma, low, high, size):
    out = []
    while sum(len(x) for x in out) < size:
        x = rng.normal(mean, sigma, size)
        out.append(x[(x >= low) & (x <= high)])
    return np.concatenate(out)[:size]

def truncated_exponential(slope, low, high, size):
    u = rng.random(size)
    if abs(slope) < 1e-12:
        return low + (high-low)*u
    a, b = np.exp(slope*low), np.exp(slope*high)
    return np.log(a + u*(b-a))/slope

edges = np.linspace(0,1,11)
sig_bdt_values = np.array([0.05,0.08,0.12,0.20,0.35,0.60,0.95,1.35,1.75,2.10])
bkg_bdt_values = np.array([2.20,1.80,1.35,0.95,0.65,0.40,0.25,0.15,0.08,0.04])

def histogram_sample(values, size):
    widths=np.diff(edges)
    probs=values*widths
    probs=probs/probs.sum()
    bins=rng.choice(len(values),size=size,p=probs)
    return edges[bins] + rng.random(size)*widths[bins]

mass = np.concatenate((
    truncated_normal(5.279,0.014,5.20,5.35,ns),
    truncated_exponential(-8.0,5.20,5.35,nb),
))
bdt = np.concatenate((
    histogram_sample(sig_bdt_values,ns),
    histogram_sample(bkg_bdt_values,nb),
))
perm=rng.permutation(N)
data=data.take(jnp.asarray(perm))
mass=jnp.asarray(mass[perm])
bdt=jnp.asarray(bdt[perm])

plot_dalitz(data,x="s13",y="s23",title="Joint-fit Dalitz sample")
plt.show()


In [ ]:
d=data.as_dict()
signal_dp=model.pdf()
mass_mean=Parameter("mass_mean",5.270,bounds=(5.24,5.31),step=0.001)

sig_mass=Gaussian1D(mass_mean,0.014,5.20,5.35)
bkg_mass=Exponential1D(-8.0,5.20,5.35)
sig_bdt=Histogram1D(jnp.asarray(edges),jnp.asarray(sig_bdt_values))
bkg_bdt=Histogram1D(jnp.asarray(edges),jnp.asarray(bkg_bdt_values))

signal_full=FactorizedDensity(
    lambda v:signal_dp(d,v),
    {"mass":mass,"bdt":bdt},
    {"mass":sig_mass,"bdt":sig_bdt},
)
bkg_norm=jnp.mean(norm.weights*bkg_shape(norm.as_dict()))
bkg_dp=bkg_shape(d)/bkg_norm
bkg_full=bkg_dp*bkg_mass(mass)*bkg_bdt(bdt)

f_sig=Parameter("signal_fraction",0.60,bounds=(0.01,0.99),step=0.01)
nll=MultiBackgroundNLL(
    signal_density=signal_full,
    backgrounds=(BackgroundCategory("combinatorial",bkg_full,1.0),),
    signal_fraction=f_sig,
)
result=Minimizer(nll,(f_sig,mass_mean)).fit(
    start_values={"signal_fraction":0.60,"mass_mean":5.270},
    simplex=True,ncall=20_000,
)
fit={p.name:float(result.values[p.name]) for p in (f_sig,mass_mean)}
print("valid:",result.valid)
print("signal fraction true / fit:",FS_TRUE,fit["signal_fraction"])
print("mass mean true / fit:",5.279,fit["mass_mean"])


In [ ]:
fs=fit["signal_fraction"]

mass_edges=np.linspace(5.20,5.35,61)
mass_centers=0.5*(mass_edges[:-1]+mass_edges[1:])
mass_widths=np.diff(mass_edges)
mass_pdf=fs*np.asarray(sig_mass(jnp.asarray(mass_centers),fit))+(1-fs)*np.asarray(bkg_mass(jnp.asarray(mass_centers)))
mass_expected=N*mass_widths*mass_pdf

fig,ax=plt.subplots(figsize=(7,5))
plot_binned_data(np.asarray(mass),bins=mass_edges,ax=ax,label="toy data")
ax.stairs(mass_expected,mass_edges,label="fitted marginal PDF")
ax.set_xlabel(r"$m_B$ [GeV]")
ax.set_ylabel("events / bin")
ax.legend()
plt.show()

bdt_centers=0.5*(edges[:-1]+edges[1:])
bdt_widths=np.diff(edges)
bdt_pdf=fs*np.asarray(sig_bdt(jnp.asarray(bdt_centers),fit))+(1-fs)*np.asarray(bkg_bdt(jnp.asarray(bdt_centers),fit))
bdt_expected=N*bdt_widths*bdt_pdf

fig,ax=plt.subplots(figsize=(7,5))
plot_binned_data(np.asarray(bdt),bins=edges,ax=ax,label="toy data")
ax.stairs(bdt_expected,edges,label="fitted marginal PDF")
ax.set_xlabel("BDT")
ax.set_ylabel("events / bin")
ax.legend()
plt.show()
